# Day 066 — Exercise 2: Image to Bytes

**What you'll build:** `image_to_bytes(img, format)` — serialize a PIL Image to raw bytes using BytesIO.

**Why it matters:** Converting images to bytes is the gateway to every downstream operation — sending images over HTTP, encoding as base64 for a vision model, or storing in a database. The RGBA→RGB handling before JPEG is real production code you will hit every time users upload transparent PNGs.

In [ ]:
import io
from PIL import Image

# Test images
_rgb  = Image.new('RGB',  (100, 80), color=(50, 100, 200))
_rgba = Image.new('RGBA', (60, 60),  color=(0, 200, 100, 128))


## Task

Implement `image_to_bytes(img, format='PNG') -> bytes`:

1. Create `io.BytesIO()`
2. If `format` is `'JPEG'` or `'JPG'` and `img.mode` is `'RGBA'` or `'P'`, convert to `'RGB'` first
3. Call `.save(buf, format=format)`
4. Return `buf.getvalue()`

**Magic bytes reference:**
- PNG: starts with `b'\x89PNG'`
- JPEG: starts with `b'\xff\xd8'` (SOI marker)

## Your Implementation

In [ ]:
def image_to_bytes(img: Image.Image, format: str = 'PNG') -> bytes:
    """Serialize a PIL Image to bytes in the given format.

    For JPEG output, RGBA and P mode images are automatically converted to
    RGB before saving (JPEG does not support alpha channels).

    Args:
        img:    PIL Image to encode
        format: format string — 'PNG', 'JPEG', 'BMP', etc.
    Returns: raw bytes of the encoded image
    """
    raise NotImplementedError


In [ ]:
def image_to_bytes(img: Image.Image, format: str = 'PNG') -> bytes:
    buf = io.BytesIO()
    out = img
    if format.upper() in ('JPEG', 'JPG') and img.mode in ('RGBA', 'P'):
        out = img.convert('RGB')
    out.save(buf, format=format)
    return buf.getvalue()


## Automated checks

In [ ]:
score, total = 0, 5
try:
    result = image_to_bytes(_rgb, 'PNG')
    assert isinstance(result, bytes), f"Expected bytes, got {type(result)}"
    score += 1; print("\u2705 returns bytes")

    assert len(result) > 0, "bytes should not be empty"
    score += 1; print("\u2705 bytes are non-empty")

    # PNG magic bytes: \x89PNG
    assert result[:4] == b'\x89PNG', (
        f"Expected PNG magic bytes, got {result[:4]!r}")
    score += 1; print("\u2705 PNG output has correct magic bytes (\\x89PNG)")

    jpeg_bytes = image_to_bytes(_rgb, 'JPEG')
    assert jpeg_bytes[:2] == b'\xff\xd8', (
        f"Expected JPEG SOI marker, got {jpeg_bytes[:2]!r}")
    score += 1; print("\u2705 JPEG output has correct SOI marker (\\xff\\xd8)")

    # RGBA -> JPEG should work without error (auto-convert to RGB)
    jpeg_from_rgba = image_to_bytes(_rgba, 'JPEG')
    assert len(jpeg_from_rgba) > 0
    score += 1; print("\u2705 RGBA image converts to JPEG without error")

except Exception as e:
    print(f"\u274c {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python
def image_to_bytes(img: Image.Image, format: str = 'PNG') -> bytes:
    buf = io.BytesIO()
    out = img
    if format.upper() in ('JPEG', 'JPG') and img.mode in ('RGBA', 'P'):
        out = img.convert('RGB')
    out.save(buf, format=format)
    return buf.getvalue()
```

**Why check magic bytes?** Magic bytes are the first few bytes of a file that identify its format — independent of the file extension. Checking them in tests proves the encoding actually happened, not just that bytes were returned. PNG's magic is `\x89PNG\r\n\x1a\n`. JPEG's is `\xff\xd8` (Start of Image marker).

</details>